In [1]:
import os
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ==========================================
# 1. 파일 경로 설정 및 데이터 불러오기
# ==========================================
base_path = "/Users/minsoo/Downloads/Gmail"

adls_path = os.path.join(base_path, "ADLS_PDS2019.csv")
adsl_path = os.path.join(base_path, "ADSL_PDS2019.csv")

# 파일 로드 (엑셀 파일일 경우 pd.read_excel 사용)
print("데이터를 불러오는 중...")
adls = pd.read_csv(adls_path)
adsl = pd.read_csv(adsl_path)

# ==========================================
# 2. 표적 병변 전처리 및 SLD(직경 총합) 계산
# ==========================================
print("전처리 진행 중...")

## 1. 표적 병변 & 단일 판독의(Radiologist 1)만 필터링 (중복 판독 제거)
target_lesions = adls[
    (adls['LSCAT'] == 'Target lesion') & 
    (adls['LSREADER'] == 'Radiologist 1')
].copy()

# 2. 환자(SUBJID)와 방문(VISIT)으로만 묶어서 합산! (VISITDY가 달라도 같은 Week면 하나로 합침)
sld_data = target_lesions.groupby(['SUBJID', 'VISIT']).agg({
    'LSLD': 'sum',          # 그 주차의 모든 표적 종양 크기를 완벽히 합산
    'VISITDY': 'mean'       # 경과 일수는 평균값으로 대표값 지정
}).reset_index()
sld_data.rename(columns={'LSLD': 'SLD'}, inplace=True)

# 3. 기저 시점(Screening) SLD를 환자당 무조건 '단 1개'로 추출
base_df = sld_data[sld_data['VISIT'] == 'Screening'][['SUBJID', 'SLD']].copy()
base_df = base_df.groupby('SUBJID')['SLD'].mean().reset_index() # 혹시 모를 중복 방지
base_df.rename(columns={'SLD': 'BASE'}, inplace=True)

# 4. 기저치 결합 및 변화량(CHG) 계산
analysis_df = pd.merge(sld_data, base_df, on='SUBJID', how='inner')
analysis_df['CHG'] = analysis_df['SLD'] - analysis_df['BASE']

# 5. Screening 제외 및 정규 주차(Week) 데이터만 필터링
final_df = analysis_df[
    (analysis_df['VISIT'] != 'Screening') & 
    (analysis_df['VISIT'].str.contains('Week', na=False))
].copy()

# 6. ADSL 병합 (환자 정보 매핑)
adsl_sub = adsl[['SUBJID', 'TRT', 'AGE', 'SEX', 'B_ECOG']].drop_duplicates(subset=['SUBJID'])
final_mmrm_clean = pd.merge(final_df, adsl_sub, on='SUBJID', how='inner')

# 정렬
final_mmrm_clean.sort_values(by=['SUBJID', 'VISITDY'], inplace=True)

# 저장
output_path = "/Users/minsoo/Downloads/Gmail/mmrm_processed_dataset.csv"
final_mmrm_clean.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f" 정제 완료! 중복이 완벽히 제거되었습니다:\n-> {output_path}")

# 중복 검증 (결과가 0이어야 완벽함)
dup_count = final_mmrm_clean.duplicated(subset=['SUBJID', 'VISIT']).sum()
print(f"환자별 동일 방문 중복 행 개수: {dup_count}개 (0개이면 성공!)")

# ==========================================
# 5. [수정됨] 데이터 정제 및 안정적인 MMRM 실행
# ==========================================
print("\nMMRM 데이터 정제 및 분석 준비 중...")

# (1) 분석에 필요한 필수 컬럼 결측치 제거
analysis_ready_df = final_mmrm_clean.dropna(subset=['CHG', 'BASE', 'TRT', 'VISIT']).copy()

# (2) 방문별 환자 수 확인 (각 방문 회차에 최소 5명 이상 있는 정규 방문만 필터링)
# 'Unscheduled' 같은 비정기 방문이나 환자가 1명뿐인 후반부 방문 제거
visit_counts = analysis_ready_df['VISIT'].value_counts()
valid_visits = visit_counts[visit_counts >= 5].index.tolist()

# 'Screening' 제외 및 유효 방문만 유지
valid_visits = [v for v in valid_visits if v != 'Screening']
analysis_ready_df = analysis_ready_df[analysis_ready_df['VISIT'].isin(valid_visits)].copy()

print("\n[분석에 포함된 방문 회차별 환자 수 (TRT별)]")
print(pd.crosstab(analysis_ready_df['VISIT'], analysis_ready_df['TRT']))

# (3) 모델 안정화를 위해 범주형 타입 명시
analysis_ready_df['TRT'] = analysis_ready_df['TRT'].astype('category')
analysis_ready_df['VISIT'] = analysis_ready_df['VISIT'].astype('category')

# (4) MMRM 실행
print("\nMMRM 모델 적합(Fit) 진행 중...")
try:
    # 1안: 표준 범주형 상호작용 모델
    model = smf.mixedlm(
        formula="CHG ~ BASE + C(TRT) + C(VISIT) + C(TRT):C(VISIT)",
        data=analysis_ready_df,
        groups=analysis_ready_df["SUBJID"]
    ).fit(method='bfgs') # linalg 에러 방지를 위해 최적화 알고리즘 지정
    
    print("\n" + "="*50)
    print(" MMRM 분석 결과 요약 (Summary)")
    print("="*50)
    print(model.summary())

except Exception as e:
    print(f"\n범주형 모델 수렴 실패로 연속형 시간(VISITDY) 모형으로 대체 실행합니다. (이유: {e})")
    # 2안: 시계열(경과 일수)을 연속형으로 본 매우 안정적인 Random Slope 모델
    model_cont = smf.mixedlm(
        formula="CHG ~ BASE + C(TRT) * VISITDY",
        data=analysis_ready_df,
        groups=analysis_ready_df["SUBJID"]
    ).fit()
    print(model_cont.summary())

데이터를 불러오는 중...
전처리 진행 중...
 정제 완료! 중복이 완벽히 제거되었습니다:
-> /Users/minsoo/Downloads/Gmail/mmrm_processed_dataset.csv
환자별 동일 방문 중복 행 개수: 0개 (0개이면 성공!)

MMRM 데이터 정제 및 분석 준비 중...

[분석에 포함된 방문 회차별 환자 수 (TRT별)]
TRT      Best supportive care  panit. plus best supportive care
VISIT                                                          
Week 12                     8                                40
Week 16                     4                                25
Week 24                     3                                17
Week 32                     2                                 6
Week 8                     68                                99

MMRM 모델 적합(Fit) 진행 중...

 MMRM 분석 결과 요약 (Summary)
                                   Mixed Linear Model Regression Results
Model:                              MixedLM                  Dependent Variable:                  CHG       
No. Observations:                   272                      Method:                              REML      
No. Group

In [3]:
import pandas as pd
import statsmodels.formula.api as smf

# 1. 방금 완벽하게 저장된 깨끗한 CSV 파일을 직접 로드
clean_csv_path = "/Users/minsoo/Downloads/Gmail/mmrm_processed_dataset.csv"
df = pd.read_csv(clean_csv_path)

# 2. 환자가 최소 5명 이상 남아있는 정규 주차만 선택 (Week 8, Week 12, Week 16, Week 24)
visit_counts = df['VISIT'].value_counts()
valid_visits = visit_counts[visit_counts >= 5].index.tolist()
model_df = df[df['VISIT'].isin(valid_visits)].copy()

# 3. 대조군을 기준(Reference)으로 범주형 설정
model_df['TRT'] = pd.Categorical(model_df['TRT'], categories=['Best supportive care', 'panit. plus best supportive care'])
model_df['VISIT'] = model_df['VISIT'].astype(str)

print("[최종 분석 대상 방문별 환자 분포 (정규 주차만 남음)]")
print(pd.crosstab(model_df['VISIT'], model_df['TRT']))

# 4. MMRM 모델 실행
final_model = smf.mixedlm(
    formula="CHG ~ BASE + C(TRT) + C(VISIT) + C(TRT):C(VISIT)",
    data=model_df,
    groups=model_df["SUBJID"]
).fit(method='bfgs')

# 5. 잘림 없이 전체 결과 출력
print("\n" + "="*70)
print(" [최종 MMRM 분석 결과: 파니투무맙의 종양 축소 효과 검증] ")
print("="*70)

result_df = pd.DataFrame({
    '계수(Coef)': final_model.params,
    '표준오차(Std.Err)': final_model.bse,
    'P-value': final_model.pvalues
})
result_df['유의성'] = result_df['P-value'].apply(
    lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
)
print(result_df.round(4))

[최종 분석 대상 방문별 환자 분포 (정규 주차만 남음)]
TRT      Best supportive care  panit. plus best supportive care
VISIT                                                          
Week 12                     8                                40
Week 16                     4                                25
Week 24                     3                                17
Week 32                     2                                 6
Week 8                     68                                99

 [최종 MMRM 분석 결과: 파니투무맙의 종양 축소 효과 검증] 
                                                    계수(Coef)  표준오차(Std.Err)  \
Intercept                                            26.3210        14.4072   
C(TRT)[T.panit. plus best supportive care]          -30.2808        14.6138   
C(VISIT)[T.Week 16]                                   6.0269        14.7973   
C(VISIT)[T.Week 24]                                   2.2445        16.5398   
C(VISIT)[T.Week 32]                                  10.3224        19.4033   
C(VISI

In [4]:
# 주차별 환자의 공식 판정(RSRESP) 흐름 확인 
import os
import pandas as pd

base_path = os.path.expanduser("~/Downloads/Gmail")
adrsp = pd.read_csv(os.path.join(base_path, "ADRSP_PDS2019.csv"))
adsl = pd.read_csv(os.path.join(base_path, "ADSL_PDS2019.csv"))

# 1. 단일 판독의(Radiologist 1) 기준 데이터 정제
adrsp_r1 = adrsp[adrsp['RSREADER'] == 'Radiologist 1'].copy()
full_journey = pd.merge(adrsp_r1, adsl[['SUBJID', 'TRT']], on='SUBJID', how='inner')

# 2. 정규 주차(Week 8, 12, 16, 24, 32, 40)만 필터링
regular_weeks = ['Week 8', 'Week 12', 'Week 16', 'Week 24', 'Week 32', 'Week 40']
full_journey = full_journey[full_journey['VISIT'].isin(regular_weeks)]

# 3. 주차별 x 치료군별 환자의 공식 판정(RSRESP) 흐름표 생성
journey_crosstab = pd.crosstab(
    [full_journey['VISIT'], full_journey['RSRESP']], 
    full_journey['TRT'], 
    margins=True
)

print("="*75)
print(" [주차별 연속 추적] 치료 경과에 따른 환자 반응 상태(RSRESP)의 변화 ")
print("="*75)
print(journey_crosstab)

 [주차별 연속 추적] 치료 경과에 따른 환자 반응 상태(RSRESP)의 변화 
TRT                          Best supportive care  \
VISIT   RSRESP                                      
Week 12 Partial response                        0   
        Progressive disease                     6   
        Stable disease                          3   
        Unable to evaluate                      0   
        Unknown                                 1   
Week 16 Partial response                        0   
        Progressive disease                     1   
        Stable disease                          3   
        Unable to evaluate                      0   
        Unknown                                 1   
Week 24 Partial response                        0   
        Progressive disease                     2   
        Stable disease                          1   
Week 32 Partial response                        0   
        Progressive disease                     2   
Week 40 Partial response                        0   
 